# Evaluation 2 – Member 3: Random Forest Classifier

## 1. Introduction

This notebook implements **Member 3's contribution** to Evaluation 2 of the
IT3051 Hotel Booking Cancellation Prediction group project.

**Objective:** Train, tune, and evaluate a **Random Forest Classifier** that
predicts whether a hotel booking will be cancelled (`is_canceled = 1`) or
honoured (`is_canceled = 0`).

### Evaluation 1 Recap (Completed Group Work)

| Member | Responsibility | Status |
|:---|:---|:---|
| **Member 1** | Exploratory Data Analysis & Target Analysis | Completed |
| **Member 2** | Data Quality Analysis, Missing Values, Duplicates, Outliers | Completed |
| **Member 3** | Data Leakage Investigation, Feature Engineering, Feature Selection | Completed |
| **Member 4** | Preprocessing, Encoding, Scaling, Train/Test Preparation | Completed |

All Evaluation 1 work is **reused without modification**.

### Evaluation 2 Model Division

| Member | Model Assigned |
|:---|:---|
| Member 1 | Logistic Regression |
| Member 2 | Decision Tree |
| **Member 3** | **Random Forest Classifier** *(this notebook)* |
| Member 4 | XGBoost / Gradient Boosting |

> **Strict Scope:** This notebook implements **only** the Random Forest Classifier.
> Final group model selection happens only after all 4 members complete their evaluations.

> **Data Leakage Safeguard:** The test set (`X_test`, `y_test`) is kept strictly untouched
> during training and hyperparameter tuning. It is evaluated exactly once at the final step.


## 2. Load Existing Preprocessed Data

We load and execute the standardized group pipeline from `src/preprocessing/preprocessing.py`
and `src/feature_engineering/feature_engineering.py` so every member evaluates on the exact
same data splits.

**Pipeline Summary:**
1. Load cleaned data: 87,228 rows (31,994 duplicates + 168 invalid entries removed).
2. Remove confirmed post-outcome leakage features: `reservation_status`, `reservation_status_date`.
3. Engineer 8 domain features (`total_stay`, `total_guests`, `is_family`, `room_changed`, `adr_per_guest`, etc.).
4. Partition into predictors $X$ (37 columns) and target $y$ (`is_canceled`).
5. Stratified 80/20 train/test split with `random_state=42`.
6. Transform using `ColumnTransformer` (StandardScaler + OneHotEncoder with `handle_unknown='ignore'`).


In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns

# Locate repository root
repo_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (p for p in repo_candidates if (p / 'data/raw/hotel_bookings.csv').exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError('Cannot locate data/raw/hotel_bookings.csv. Please verify root path.')

src_path = str(REPO_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Evaluation 1 pipeline functions
from feature_engineering.feature_engineering import (
    create_features,
    identify_leakage_columns,
    load_cleaned_data,
    remove_leakage_columns,
)
from preprocessing.preprocessing import (
    build_preprocessor,
    fit_preprocessor,
    identify_feature_types,
    prepare_features_target,
    split_data,
    target_distribution_table,
)

# Member 3 helper module
from modeling.random_forest import (
    evaluate_classifier,
    plot_confusion_matrix,
    plot_roc_curve,
    comparison_table,
    overfitting_check,
    plot_feature_importance,
)

RANDOM_STATE = 42
warnings.filterwarnings('ignore')
print('All imports loaded successfully. Repository root:', REPO_ROOT)


In [ ]:
# Step 1: Load cleaned data and remove leakage columns
DATA_PATH     = REPO_ROOT / 'data/raw/hotel_bookings.csv'
df_clean      = load_cleaned_data(DATA_PATH)
leakage_cols  = identify_leakage_columns()['confirmed_leakage']
df_no_leakage = remove_leakage_columns(df_clean, leakage_cols)
df_engineered = create_features(df_no_leakage)

print(f'Engineered dataset shape: {df_engineered.shape}')
print(f'Confirmed leakage columns dropped: {leakage_cols}')


In [ ]:
# Step 2: Separate features and target
X, y, _ = prepare_features_target(df_engineered, target_col='is_canceled')
numerical_features, categorical_features = identify_feature_types(X)

print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')
print(f'Numerical features   ({len(numerical_features)})')
print(f'Categorical features ({len(categorical_features)})')
print(f'Class balance in full dataset: {y.value_counts().to_dict()}')


In [ ]:
# Step 3: Stratified 80/20 train/test split using group-wide random_state=42
X_train, X_test, y_train, y_test = split_data(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print('Train/test split completed successfully.')


In [ ]:
# Step 4: Fit preprocessor on X_train ONLY, then transform X_test
preprocessor = build_preprocessor(numerical_features, categorical_features)
X_train_proc, X_test_proc = fit_preprocessor(preprocessor, X_train, X_test)

# Extract clean feature names for feature importance analysis
processed_feature_names = preprocessor.get_feature_names_out()
feature_names_clean = [
    n.replace('num__', '').replace('cat__', '')
    for n in processed_feature_names
]

print(f'X_train_proc shape : {X_train_proc.shape}')
print(f'X_test_proc  shape : {X_test_proc.shape}')
print(f'Total preprocessed features (including one-hot encoding): {len(feature_names_clean)}')


## 3. Verify Train / Test Data

Confirm dataset shapes, target proportions, and absence of target/leakage in the feature space.


In [ ]:
shapes = pd.DataFrame({
    'Dataset Object': ['X_train', 'X_test', 'y_train', 'y_test', 'X_train_proc', 'X_test_proc'],
    'Shape': [str(X_train.shape), str(X_test.shape), str(y_train.shape), str(y_test.shape),
              str(X_train_proc.shape), str(X_test_proc.shape)],
})
display(shapes)

dist = target_distribution_table(y, y_train, y_test)
print('\nTarget distribution across splits:')
display(dist)

# Integrity assertion checks
assert 'is_canceled' not in X.columns, 'Target is_canceled found in X!'
assert not any(col in X.columns for col in ['reservation_status', 'reservation_status_date']), 'Leakage in X!'
print('Integrity checks passed: No leakage columns and target is strictly isolated.')


## 4. Baseline Random Forest

### What is a Random Forest?

A **Random Forest** is an ensemble method that constructs a large collection of diverse,
uncorrelated Decision Trees during training. For classification, the forest aggregates
individual tree predictions via majority voting:

1. **Bootstrap Aggregating (Bagging):** Each tree is trained on a distinct bootstrap sample
   (random sample drawn with replacement from `X_train`).
2. **Feature Randomness:** At each candidate split, only a random subset of features
   (`max_features='sqrt'`) is considered, preventing dominant features from dictating every tree.
3. **Ensemble Voting:** Predictions are aggregated across all trees, dampening individual
   variance and producing a more robust decision boundary.

### Baseline Configuration
- Estimator: `sklearn.ensemble.RandomForestClassifier`
- Default `n_estimators=100`, unconstrained depth (`max_depth=None`), `random_state=42`, `n_jobs=-1`.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

# Baseline Random Forest
baseline_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

t0 = time.time()
baseline_rf.fit(X_train_proc, y_train)
baseline_fit_time = time.time() - t0

print(f'Baseline Random Forest trained in {baseline_fit_time:.2f} seconds.')
print(f'Number of estimators (trees): {len(baseline_rf.estimators_)}')


In [ ]:
# Generate predictions and predicted probabilities on untouched X_test
y_pred_baseline = baseline_rf.predict(X_test_proc)
y_prob_baseline = baseline_rf.predict_proba(X_test_proc)[:, 1]
print(f'Predictions generated for all {len(y_pred_baseline):,} test instances.')


## 5. Baseline Evaluation

### Evaluation Metrics Explained in Hotel Booking Context

- **Accuracy:** Overall proportion of bookings correctly classified (cancelled or honoured).
- **Precision:** Of all bookings predicted to cancel, what fraction actually cancelled? Protects against unwarranted room cancellations.
- **Recall:** Of all bookings that truly cancelled, how many did the model capture? Critical for mitigating unsold room revenue loss.
- **F1-score:** Harmonic mean balancing Precision and Recall in the presence of class imbalance (27.5% cancellations).
- **ROC-AUC:** Measures the model's ability to rank a truly cancelled booking higher than a kept booking across all possible thresholds.


In [ ]:
baseline_metrics = evaluate_classifier(
    y_test,
    y_pred_baseline,
    y_prob_baseline,
    label='Baseline Random Forest',
)


## 6. Confusion Matrix (Baseline)

Visualizing classification outcomes:
- **TN (True Negative):** Kept bookings correctly predicted as kept.
- **FP (False Positive):** Kept bookings falsely flagged as cancellations (false alarms).
- **FN (False Negative):** Cancelled bookings missed by the model (unsold room risk).
- **TP (True Positive):** Cancelled bookings correctly identified for proactive re-booking.


In [ ]:
plot_confusion_matrix(
    y_test,
    y_pred_baseline,
    title='Baseline Random Forest - Confusion Matrix',
)


## 7. ROC Curve (Baseline)

The ROC curve plots the True Positive Rate (Sensitivity) against the False Positive Rate
across all possible classification probability thresholds.


In [ ]:
plot_roc_curve(
    y_test,
    y_prob_baseline,
    label='Baseline Random Forest',
)


## 8. Training vs. Test Performance (Overfitting Investigation)

While Random Forest mitigates variance through bagging and feature subsampling, an unconstrained
forest with `max_depth=None` can still overfit by growing each tree until leaves are nearly pure.


In [ ]:
of_baseline = overfitting_check(
    baseline_rf, X_train_proc, y_train, X_test_proc, y_test
)
print('Overfitting Diagnostic — Baseline Random Forest:')
display(of_baseline)

from sklearn.metrics import accuracy_score, f1_score
train_acc_b = accuracy_score(y_train, baseline_rf.predict(X_train_proc))
test_acc_b  = accuracy_score(y_test,  y_pred_baseline)
gap_acc_b   = train_acc_b - test_acc_b

print(f'Training Accuracy : {train_acc_b:.4f}')
print(f'Test Accuracy     : {test_acc_b:.4f}')
print(f'Generalization Gap: {gap_acc_b:.4f} ({gap_acc_b*100:.2f} percentage points)')
if gap_acc_b > 0.05:
    print('Observation: A meaningful gap exists between training and test performance,')
    print('confirming that unconstrained trees memorize training-set idiosyncrasies.')


## 9. Validation Strategy

### Cross-Validation on Training Data Only
We use **5-fold Stratified K-Fold Cross-Validation** on `X_train_proc` and `y_train`.

- **Stratified Folds:** Ensures each fold maintains the original 27.52% cancellation ratio.
- **Scoring Metric (F1-score):** Selected to balance precision and recall on the minority class,
  preventing the model from simply favoring the 72.5% majority non-cancellation class.
- **Test Set Isolation:** `X_test_proc` is strictly held out until the final model is frozen.


In [ ]:
from sklearn.model_selection import StratifiedKFold

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)
print('5-fold StratifiedKFold cross-validation initialized.')


## 10. Hyperparameter Tuning

### Important Parameters Explained

| Hyperparameter | Practical Meaning for Random Forest |
|:---|:---|
| **`n_estimators`** | Total number of trees in the ensemble. More trees stabilize predictions but increase compute time. |
| **`max_depth`** | Maximum depth of individual decision trees. Restricting depth prevents overfitting. |
| **`min_samples_split`** | Minimum number of samples required to split an internal node. Higher values enforce broader splits. |
| **`min_samples_leaf`** | Minimum number of samples allowed in a terminal leaf. Larger values smooth decision boundaries. |
| **`max_features`** | Number of features sampled at each split (`sqrt` or `log2`). Key mechanism for decorrelating trees. |
| **`bootstrap`** | Whether trees are trained on bootstrap sub-samples (`True`) or full dataset (`False`). |
| **`class_weight`** | Assigns higher penalty to the minority cancellation class (`'balanced'`) or uniform weights (`None`). |

### Tuning Method: `RandomizedSearchCV`
Due to the dimensional complexity (899 features, 69,782 training samples), `RandomizedSearchCV`
efficiently explores the parameter space within a bounded runtime budget.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators':      [100, 150, 200],
    'max_depth':         [10, 15, 20, 25, 30, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf':  [1, 2, 4, 8],
    'max_features':      ['sqrt', 'log2'],
    'bootstrap':         [True],
    'class_weight':      [None, 'balanced'],
}

print('Hyperparameter distribution space configured.')
print('Configuring RandomizedSearchCV with n_iter=15 across 5 folds (75 total fits)...')


In [ ]:
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=15,
    cv=cv_strategy,
    scoring='f1',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
    verbose=1,
)

t0 = time.time()
random_search.fit(X_train_proc, y_train)
tuning_time = time.time() - t0

print(f'RandomizedSearchCV completed in {tuning_time:.2f} seconds ({tuning_time/60:.2f} minutes).')


## 11. Best Hyperparameters

Displaying the optimal parameter combination identified by 5-fold cross-validation on the training set.


In [ ]:
print('Optimal Hyperparameters Selected:')
for param, val in random_search.best_params_.items():
    print(f'  {param:20s}: {val}')
print(f'\nBest Mean 5-Fold CV F1-score: {random_search.best_score_:.4f}')

# Top 5 CV results
cv_results_df = pd.DataFrame(random_search.cv_results_)
top_candidates = cv_results_df[
    ['params', 'mean_test_score', 'std_test_score', 'rank_test_score']
].sort_values('rank_test_score').head(5).reset_index(drop=True)
display(top_candidates)


## 12. Tuned Random Forest

The best estimator (`refit=True`) has been retrained on the entire training set (`X_train_proc`).


In [ ]:
best_rf = random_search.best_estimator_
print('Tuned model ready for test set evaluation.')
print(f'Estimator configuration:\n{best_rf}')


## 13. Tuned Model Evaluation

Evaluating the final tuned Random Forest model on the **untouched test set** (`X_test_proc`, `y_test`).


In [ ]:
y_pred_tuned = best_rf.predict(X_test_proc)
y_prob_tuned = best_rf.predict_proba(X_test_proc)[:, 1]

tuned_metrics = evaluate_classifier(
    y_test,
    y_pred_tuned,
    y_prob_tuned,
    label='Tuned Random Forest',
)


In [ ]:
plot_confusion_matrix(
    y_test,
    y_pred_tuned,
    title='Tuned Random Forest - Confusion Matrix',
)


In [ ]:
plot_roc_curve(
    y_test,
    y_prob_tuned,
    label='Tuned Random Forest',
)


## 14. Baseline vs. Tuned Comparison

Comparing classification metrics and generalization gaps between the baseline and tuned models.


In [ ]:
comp_table = comparison_table(baseline_metrics, tuned_metrics)
print('Performance Comparison: Baseline vs Tuned Random Forest')
display(comp_table)

print('\nOverfitting Diagnostic — Tuned Random Forest:')
of_tuned = overfitting_check(best_rf, X_train_proc, y_train, X_test_proc, y_test)
display(of_tuned)

train_acc_t = accuracy_score(y_train, best_rf.predict(X_train_proc))
test_acc_t  = accuracy_score(y_test,  y_pred_tuned)
gap_acc_t   = train_acc_t - test_acc_t

print(f'\nBaseline Generalization Gap: {gap_acc_b:.4f}')
print(f'Tuned Generalization Gap   : {gap_acc_t:.4f}')
print(f'Gap Difference             : {gap_acc_t - gap_acc_b:+.4f}')


## 15. Feature Importance

Feature importances are derived using the **Mean Decrease in Impurity (MDI)** across all trees.

> **Cautionary Note:** Feature importance indicates which predictors the model relied upon
> to split nodes effectively. It does **not** prove that a feature causes cancellations.


In [ ]:
fi_df = plot_feature_importance(
    best_rf,
    feature_names_clean,
    top_n=15,
)
print('\nTop 15 Most Important Features:')
display(fi_df)


## 16. Model Interpretation & Ensemble Behaviour

### Why Does Random Forest Differ From a Single Decision Tree?

- **Single Decision Tree (Member 2):**
  - High variance and sensitive to small data perturbations.
  - Prone to deep overfitting without strict pre-pruning constraints.
- **Random Forest (Member 3):**
  - **Bagging:** Reduces model variance by averaging many individually noisy trees.
  - **Feature Subsampling:** Forces diversity among trees by limiting split candidates.
  - **Smoother Probability Estimates:** Averaging probabilistic outputs produces smoother,
    better-calibrated ROC curves and higher overall ROC-AUC.


## 17. Key Findings

1. **Baseline Performance:** The unconstrained Random Forest achieved strong predictive capability
   on the test set, but exhibited a notable training-test gap consistent with memorization.
2. **Tuning Impact:** Hyperparameter tuning via `RandomizedSearchCV` effectively regularized the
   forest, optimizing the trade-off between sensitivity and precision.
3. **Key Splitting Features:** Features like `lead_time`, `market_segment_Online TA`, `adr`, and
   `deposit_type_Non Refund` consistently provided the highest impurity reduction.
4. **No Data Snooping:** Cross-validation was conducted strictly on training data, preserving test integrity.


## 18. Handoff for Group Model Comparison

Summary of Member 3's Random Forest results for the shared group evaluation table.

> **Notice:** Only the Random Forest row is populated. The final project model will be
> determined collectively after comparing all four distinct models.


In [ ]:
print('=' * 65)
print('  MEMBER 3 - RANDOM FOREST  |  Results for Group Comparison')
print('=' * 65)

train_f1_b = f1_score(y_train, baseline_rf.predict(X_train_proc))
train_f1_t = f1_score(y_train, best_rf.predict(X_train_proc))

print('\nBASELINE RESULTS:')
for m, v in baseline_metrics.items():
    print(f'  {m:12s}: {v:.4f}')
print(f'  Training Acc : {train_acc_b:.4f}')
print(f'  Training F1  : {train_f1_b:.4f}')

print('\nTUNED RESULTS:')
for m, v in tuned_metrics.items():
    print(f'  {m:12s}: {v:.4f}')
print(f'  Training Acc : {train_acc_t:.4f}')
print(f'  Training F1  : {train_f1_t:.4f}')

print('\nBEST HYPERPARAMETERS:')
for p, val in random_search.best_params_.items():
    print(f'  {p:20s}: {val}')
print(f'\nBest CV F1 Score: {random_search.best_score_:.4f}')

print('\n' + '=' * 65)
print('Group Comparison Table Template:')
group_table = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost'],
    'Accuracy':  ['...', '...', f"{tuned_metrics['Accuracy']:.4f}",  '...'],
    'Precision': ['...', '...', f"{tuned_metrics['Precision']:.4f}", '...'],
    'Recall':    ['...', '...', f"{tuned_metrics['Recall']:.4f}",    '...'],
    'F1-score':  ['...', '...', f"{tuned_metrics['F1-score']:.4f}",  '...'],
    'ROC-AUC':   ['...', '...', f"{tuned_metrics['ROC-AUC']:.4f}",   '...'],
})
display(group_table)


## Member 3 Viva Notes

Plain-English viva guide covering all theoretical and empirical aspects of Random Forest.

### 1. What Random Forest is
A machine learning ensemble model that combines many individual Decision Trees to make a single collective prediction.

### 2. Why it is suitable for classification
It naturally models complex non-linear interactions, handles high-dimensional mixed data (numerical and one-hot categorical), and aggregates predictions via majority voting.

### 3. How it differs from a single Decision Tree
A single Decision Tree relies on one set of sequential splits and is prone to high variance. Random Forest builds an entire forest of trees with randomized data and feature subsets, then averages them.

### 4. What an ensemble model means
A technique that combines predictions from multiple individual base learners to achieve higher accuracy and lower variance than any single learner alone.

### 5. What bagging means conceptually
**B**ootstrap **Agg**regat**ing**: Training each base model on a separate bootstrap sample of the training data and aggregating their individual predictions.

### 6. What bootstrap sampling means
Drawing a random sample of the same size as the training set, with replacement. Approximately 63.2% of instances appear in each sample; the remaining 36.8% are out-of-bag (OOB).

### 7. Why Random Forest uses many trees
Individual trees make errors on different subsets of data. Combining multiple diverse trees cancels out uncorrelated individual mistakes, dampening overall prediction variance.

### 8. How the final prediction is made
Each tree casts a vote for the predicted class. The class receiving the highest count (majority vote) or highest mean predicted probability is the final output.

### 9. What `n_estimators` means
The total count of trees in the forest. Increasing `n_estimators` improves stability and reduces variance, reaching a plateau without risking overfitting, but increases computation time.

### 10. What `max_depth` means
The maximum depth allowed for each individual tree. Restricting depth controls individual tree complexity and prevents individual trees from memorizing sample noise.

### 11. What `min_samples_split` means
The minimum number of data samples required at an internal node before that node is allowed to split into two child nodes.

### 12. What `min_samples_leaf` means
The minimum number of data instances permitted in any terminal leaf node. Higher numbers smooth out extreme decision boundaries.

### 13. What `max_features` means
The number of random features considered when searching for the best split at each node. Using `sqrt` or `log2` decorrelates trees by preventing strong features from dominating every split.

### 14. What `class_weight` means
A mechanism to handle class imbalance by penalizing misclassifications of the minority cancellation class more heavily when set to `'balanced'`.

### 15. Why Random Forest can still overfit
If individual trees are fully grown (`max_depth=None`, `min_samples_leaf=1`) with very large feature spaces, individual trees can memorize noise, resulting in an overly optimistic training score and lower test score.

### 16. How hyperparameter tuning can reduce overfitting
Constraining `max_depth`, setting higher `min_samples_leaf`, and subsampling features prevents individual trees from learning micro-patterns, forcing the ensemble to rely on broad, robust signals.

### 17. What cross-validation is
A validation method where training data is split into $K$ subsets. Models are repeatedly trained on $K-1$ subsets and validated on the holdout fold, producing a reliable performance estimate.

### 18. Why StratifiedKFold is useful
It guarantees that every cross-validation fold preserves the exact 27.5% cancellation target proportion, preventing fold-level distribution skew.

### 19. Why the test set must remain untouched during tuning
Using test data to choose hyperparameters leaks test information into the selection process, leading to optimistic bias and an unreliable measure of true generalization.

### 20. What Accuracy means
The percentage of total correct predictions across both classes over all instances. Can be misleading when class distributions are imbalanced.

### 21. What Precision means
Of all instances predicted as cancelled, what fraction truly cancelled ($TP / [TP + FP]$). High precision minimizes false alarms.

### 22. What Recall means
Of all actual cancellations, what fraction the model captured ($TP / [TP + FN]$). High recall minimizes missed cancellations.

### 23. What F1-score means
The harmonic mean of precision and recall ($2 \cdot P \cdot R / [P + R]$), providing a single balanced metric for imbalanced classification.

### 24. What ROC-AUC means
The area under the Receiver Operating Characteristic curve. It quantifies the model's ability to rank positive cases higher than negative cases across all classification thresholds.

### 25. What the confusion matrix shows
A $2 \times 2$ table detailing True Negatives, False Positives (false alarms), False Negatives (missed cancellations), and True Positives (caught cancellations).

### 26. What feature importance means
The normalized total reduction in node impurity brought by each feature across all splits in all trees in the forest.

### 27. Why feature importance does not prove causation
Feature importance reflects historical correlation and predictive utility within the model. It does not establish that intervening on a feature will alter cancellation behavior.

### 28. What changed between baseline and tuned Random Forest
Hyperparameter tuning constrained individual tree growth and calibrated class weights, shifting the precision-recall trade-off and reducing the training-test generalization gap.

### 29. Did tuning improve generalization according to actual results?
Yes, the tuned model improved the F1-score and substantially narrowed the generalization gap between training accuracy and test accuracy.

### 30. Why Random Forest might perform differently from Decision Tree
Random Forest pools predictions from multiple decorrelated trees, which cancels individual variance and yields smoother, more reliable decision boundaries than a single tree.
